# Озвучка книги на бесплатной видеокарте

Перед началом: **Среда выполнения → Сменить тип среды выполнения → T4 GPU → Сохранить**.

В папке `audiobook` на твоём Google Диске должны лежать пять файлов: книга `babylon.epub` и два голоса
(`voice_05.wav` + `voice_05.txt`, `voice_06.wav` + `voice_06.txt`) из папки `voices12` на рабочем столе.

Дальше нажимай ▶ у ячеек по порядку. Ячейка 4 — проверка скорости на 8 кусках, ячейка 5 — вся книга.
Если сессия оборвалась, открой блокнот заново и запусти ячейки 1–3 и 5: готовые куски лежат на Диске и пропускаются.

In [ ]:
# 1. Установка (около 2 минут)
!nvidia-smi -L
!pip install -q omnivoice num2words pypdf

In [ ]:
# 2. Доступ к Google Диску (появится окно с запросом разрешения)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Настройки и код скриптов
FOLDER = "/content/drive/MyDrive/audiobook"
BOOK = f"{FOLDER}/babylon.epub"
MAIN_VOICE = f"{FOLDER}/voice_05.wav"
SIDE_VOICE = f"{FOLDER}/voice_06.wav"
!rm -rf /content/audiobook && git clone -q https://github.com/alex1noalex2/audiobook /content/audiobook
!ls -la {FOLDER}

In [ ]:
# 4. Проверка скорости: 8 кусков (1–2 минуты после загрузки модели). Смотри «с на кусок» внизу.
!python /content/audiobook/local/book2audio_cuda.py {BOOK} --ref {MAIN_VOICE} --ref2 {SIDE_VOICE} --limit 8 --out /content/test
from IPython.display import Audio, display
import glob
for f in sorted(glob.glob('/content/test/0*.wav'))[3:6]:
    print(f); display(Audio(f))

In [ ]:
# 5. Вся книга. Части MP3 (babylon_часть01.mp3 …) появляются на Google Диске по мере готовности.
!python /content/audiobook/local/book2audio_cuda.py {BOOK} --ref {MAIN_VOICE} --ref2 {SIDE_VOICE} --out {FOLDER}/babylon_audio

In [ ]:
# 6. Склейка всех частей в один MP3 (быстро, без перекодирования). Запускай, когда ячейка 5 написала «Готово.»
import glob
parts = sorted(glob.glob(f"{FOLDER}/babylon_часть*.mp3"))
print(len(parts), "частей")
with open("/content/list.txt", "w") as f:
    for p in parts:
        f.write(f"file '{p}'\n")
!ffmpeg -y -loglevel error -f concat -safe 0 -i /content/list.txt -c copy {FOLDER}/Dva_Vavilona.mp3
!ls -la {FOLDER}/Dva_Vavilona.mp3

In [ ]:
# 7. Исправления: переозвучить только куски, чей текст изменился (границы кусков и файл fixes.txt рядом с книгой).
# После неё запусти ячейку 6 ещё раз, чтобы пересобрать общий MP3.
!python /content/audiobook/local/book2audio_cuda.py {BOOK} --ref {MAIN_VOICE} --ref2 {SIDE_VOICE} --out {FOLDER}/babylon_audio --steps 16 --redo-changed

In [ ]:
# 8. Оглавление с таймкодами для Телеграма (после ячейки 6). Результат — на экране и в файле timecodes.txt на Диске.
!python /content/audiobook/local/timecodes.py {BOOK} {FOLDER}/babylon_audio --out {FOLDER}/timecodes.txt

In [ ]:
# 9. Проба пауз между словами: один кусок в четырёх вариантах (1 = как сейчас). Темп слов не меняется.
# Номер куска можно менять: 00100 -> любой из папки babylon_audio.
!python /content/audiobook/local/try_gaps.py {FOLDER}/babylon_audio/00100.wav --out /content/gaps
from IPython.display import Audio, display
import glob
for f in sorted(glob.glob('/content/gaps/*.wav')):
    print(f); display(Audio(f))

In [ ]:
# 10. Стенд проб: 3 отрывка из разных мест книги (~200 слов) в нескольких вариантах озвучки. Файлы и оценка.md — в папке bench на Диске.
# base — как сейчас; yo — только «ё» по Silero; gaps1.4 — паузы между словами; phone — примечания тем же голосом с эффектом телефона;
# yo+gaps1.4+phone+check3 — всё вместе и проверка речью (Whisper): плохо озвученные куски переозвучиваются, до 3 попыток.
!pip install -q silero-stress
!python /content/audiobook/local/bench.py {BOOK} --ref {MAIN_VOICE} --ref2 {SIDE_VOICE} --out {FOLDER}/bench2 --at 0.25 0.5 0.75 --words 200 --variants base yo base+gaps1.4 base+phone yo+gaps1.4+phone+check3

In [ ]:
# 11. Тот же стенд на другой модели: F5-TTS_RUSSIAN (родной «+» для ударений; веса некоммерческие, CC-BY-NC). Файлы — в папке bench_f5.
# plus+plusref — ударения по Silero в тексте и в записи образца; phone0.4 — едва заметный «телефон» для примечаний; check3 — проверка речью (Whisper).
!pip install -q f5-tts silero-stress
!python /content/audiobook/local/bench.py {BOOK} --ref {MAIN_VOICE} --ref2 {SIDE_VOICE} --engine f5 --out {FOLDER}/bench_f5 --at 0.25 0.5 0.75 --variants plus+plusref+phone0.4 plus+plusref+phone0.4+check3